# Interactively Create Geometries 
## To be used in molpro

In [ ]:
import numpy as np
import ase
from ase.visualize import view
import nglview as nv
import ipywidgets
print(ipywidgets.__version__)
print(nv.__version__)


In [ ]:
def generate_molpro_geometry(atoms):
    positions = atoms.get_positions()
    symbols = atoms.get_chemical_symbols()
    N = len(symbols)
    geom_str_lines = ["geometry={",
                      f"{N}",
                     ]
    for i, elem in enumerate(symbols):
        geom_str_lines.append(
            f"{elem},{",".join([f"{p:21.17f}" for p in positions[i]])}"
        )

    geom_str_lines.append("}")

    return "\n".join(geom_str_lines)
    


## Structure of NH3
(see [nh3_geom.svg](dimers/nh3_geom.svg)]
$$
r \sin \left( \frac{107}{2} \right) = r' \frac{\sqrt{3}}{2}
$$

$$ z = \left[r^2 - r'^2 \right] $$

In [ ]:
nh3 = ase.Atoms('NH3')

# r
bond_len = 1
# r'
rp = 2*bond_len/np.sqrt(3) * np.sin(107/2 * (np.pi/180)) 
z = np.sqrt(bond_len**2 - rp**2)
x = np.sqrt(3)/2 * rp
display(rp, z, x)

positions = [
    [0, 0, 0],
    [0, rp, -z],
    [-x, -0.5*rp, -z],
    [x, -0.5*rp, -z],
]

nh3.set_positions(positions)
display(nh3.get_positions())
display(nh3.get_atomic_numbers())

In [ ]:
view = nv.show_ase(nh3)
view

In [ ]:
print(generate_molpro_geometry(nh3))

## Cu_NH3

In [ ]:
def set_cu_position(cu_nh3_atoms, r):
    all_positions = cu_nh3_atoms.get_positions()
    all_positions[0,3] = r
    cu_nh3_atoms.set_positions(all_positions)

In [ ]:
cu_nh3 = ase.Atoms('CuNH3')
cu_positions = [[0, 0, 3]]
all_positions = np.concatenate(
    [cu_positions ,nh3.get_positions()]
) 

cu_nh3.set_positions(all_positions)
view = nv.show_ase(cu_nh3)
view.clear_representations()
view.add_representation('ball+stick')
view


In [ ]:

np.reshape(view._camera_orientation, (4,4))


In [ ]:
view._camera_orientation[13] = 0
view._camera_orientation[14] = 0

In [ ]:
for i, val in enumerate(view._camera_orientation):
    m = i%4
    n = i//4
    if m == n:
        view._camera_orientation[i] = 1
    else:
        view._camera_orientation[i] = 0

In [ ]:
view

In [ ]:
img_data = view.render_image(trim=True)

In [ ]:
with open('cu_nh3.png', 'wb') as out:
    out.write(img_data.value)

In [ ]:
img_data.value